# OISST/MOM6 Sea Surface Temperature Comparison
Author: Brooke Hawkins

## Set up

In [ ]:
import xarray as xr
import numpy as np

config = {
    "mom6_path": "mom6/mom6_monthly_sst.nc",
    "oisst_path": "oisst/TS_monthly.nc",
    "mom6_var": "tos",
    "oisst_var": "sst",
    "spatial_bounds": {"lat": [32, 48], "lon": [-128, -116.8]},
    "lon_offset": 360,
    "time_bounds": {"start": None, "end": None},
}


## Load data

In [ ]:
# load datasets
sst_mom6 = xr.open_dataset(config["mom6_path"])
sst_oisst = xr.open_dataset(config["oisst_path"])


I confirmed that the MOM6 subset here has duplicates for the last six months of data, January to June of 2025. Each time step is duplicated once, and the values are identical. I need to fix this in the `sst-anomaly.ipynb` script before the data is written to netCDF.

In [ ]:
# remove duplicates from MOM6
sst_mom6 = sst_mom6.drop_duplicates(dim="time", keep="first")


## Define spatial and temporal extents

In [ ]:
# define spatial extent for California Current
lat_bounds = config["spatial_bounds"]["lat"]
lon_bounds = config["spatial_bounds"]["lon"]

# define temporal extent for comparison based on MOM6 hindcast which has a shorter timeframe than OISST
time_bgn = (
    config["time_bounds"]["start"]
    if config["time_bounds"]["start"] is not None
    else sst_mom6.time.to_index().min()
)
time_end = (
    config["time_bounds"]["end"]
    if config["time_bounds"]["end"] is not None
    else sst_mom6.time.to_index().max()
)

print(f"Date range is {time_bgn} to {time_end}")


In [ ]:
# subset both datasets (add longitude offset for 0-360 coordinate scale)
sst_mom6_subset = sst_mom6.sel(
    lat_vec=slice(lat_bounds[0], lat_bounds[1]),
    lon_vec=slice(
        lon_bounds[0] + config["lon_offset"], lon_bounds[1] + config["lon_offset"]
    ),
    time=slice(time_bgn, time_end),
)
sst_oisst_subset = sst_oisst.sel(
    lat_vec=slice(lat_bounds[0], lat_bounds[1]),
    lon_vec=slice(
        lon_bounds[0] + config["lon_offset"], lon_bounds[1] + config["lon_offset"]
    ),
    time=slice(time_bgn, time_end),
)


In [ ]:
# check grid sizes
print("MOM6 dimensions: ", sst_mom6_subset.sizes)
print("OISST dimensions: ", sst_oisst_subset.sizes)


## Check grid resolutions and interpolate

In [ ]:
def check_grid_resolution(dataset):
    """
    Analyzes the spatial resolution of a dataset's grid to determine if it is
    regular (uniform spacing), nearly regular (close to uniform spacing, some
    rounding error), or irregular.

    Args:
        dataset: An object (e.g., xarray Dataset) containing 'lat_vec' and 'lon_vec'
                 coordinate arrays that can be converted to pandas Series.
    """
    # check step size between adjacent latitude points to check for uniform spacing
    lat_series = dataset.lat_vec.to_series()
    lat_series_diff = lat_series.diff()
    lat_diff_min = lat_series_diff.min(skipna=True)
    lat_diff_max = lat_series_diff.max(skipna=True)

    # check step size between adjacent longitude points to check for uniform spacing
    lon_series = dataset.lon_vec.to_series()
    lon_series_diff = lon_series.diff()
    lon_diff_min = lon_series_diff.min(skipna=True)
    lon_diff_max = lon_series_diff.max(skipna=True)

    # classify grid resolution based on variance in step sizes
    if lat_diff_min == lat_diff_max and lon_diff_min == lon_diff_max:
        print(
            f"Perfectly regular grid resolution of {lat_diff_min} latitudinal degrees by {lon_diff_min} longitudinal degrees."
        )
    elif np.isclose(lat_diff_min, lat_diff_max) and np.isclose(
        lon_diff_min, lon_diff_max
    ):
        print(
            f"Nearly regular grid resolution of {lat_diff_min} latitudinal degrees by {lon_diff_min} longitudinal degrees."
        )
    else:
        print(
            f"Irregular grid resolution of {lat_diff_min} to {lat_diff_max} latitudinal degrees by {lon_diff_min} to {lon_diff_max} longitudinal degrees."
        )

In [ ]:
check_grid_resolution(sst_oisst_subset)
check_grid_resolution(sst_mom6_subset)

Since MOM6 has a higher resolution, I will interpolate the MOM6 data to the OISST grid. I will use bilinear interpolation using xarray's `interp_like` function, though I want to check with Darren if I should be using a more conservative approach (implement with `xESMF` library in python).

In [ ]:
# interpolate MOM6 grid onto OISST grid
sst_mom6_interp = sst_mom6_subset.interp_like(sst_oisst_subset)

# check interpolated MOM6 grid size and resolution
print("MOM6 dimensions: ", sst_mom6_interp.sizes)
check_grid_resolution(sst_mom6_interp)


Calculate weights based on latitude to area-weight statistics. The area of each 1/2 degree by 1/2 degree grid cell varies with latitude, since the distance that one degree of longitude represents shrinks when moving from the equator to the poles. If comparison statistics are not area weighted, then smaller cells further north (which are generally cooler waters) will bias comparisons.

In [ ]:
# calculate latitude-based area weights
weights = np.cos(np.deg2rad(sst_oisst_subset.lat_vec))


## Comparisons

Calculate and visualize correlation, bias, and root mean squared error (RMSE).

### Correlation

In [ ]:
# calculate correlation
mom6_data = sst_mom6_interp[config["mom6_var"]]
oisst_data = sst_oisst_subset[config["oisst_var"]]

# make sure to area weight calculation across spatial dimensions
correlation_global = xr.corr(oisst_data, mom6_data, weights=weights).item()
correlation_temporal = xr.corr(
    oisst_data, mom6_data, dim=["lat_vec", "lon_vec"], weights=weights
)
# no need to area weight calculation across time dimension
correlation_spatial = xr.corr(oisst_data, mom6_data, dim="time")


In [ ]:
# print overall correlation
correlation_global

In [ ]:
# visualize correlation over time
correlation_temporal.plot()

In [ ]:
# visualize correlation over space (heatmap)
correlation_spatial.plot()

In [ ]:
# visualize correlation over space (contour)
correlation_spatial.plot.contour()

### Bias

In [ ]:
# calculate bias
error = mom6_data - oisst_data
error_weighted = error.weighted(weights)
# make sure to area weight calculation across spatial dimensions
bias_global = error_weighted.mean().item()
bias_temporal = error_weighted.mean(dim=["lat_vec", "lon_vec"])
# no need to area weight calculation across time dimension
bias_spatial = error.mean(dim="time")


In [ ]:
# print overall bias
bias_global

In [ ]:
# visualize bias over time
bias_temporal.plot()

In [ ]:
# visualize bias over space (contour)
bias_spatial.plot.contour()

### RMSE

In [ ]:
# calculate RMSE
squared_error = error**2
squared_error_weighted = squared_error.weighted(weights)
# make sure to area weight calculation across spatial dimensions
rmse_global = np.sqrt((squared_error_weighted).mean()).item()
rmse_temporal = np.sqrt((squared_error_weighted).mean(dim=["lat_vec", "lon_vec"]))
# no need to area weight calculation across time dimension
rmse_spatial = np.sqrt((squared_error).mean(dim="time"))


In [ ]:
# print overall RMSE
rmse_global

In [ ]:
# visualize RMSE over time
rmse_temporal.plot()

In [ ]:
# visualize RMSE over space (contour)
rmse_spatial.plot.contour()